# Tarea 2: Ruteo Consistente de Vehículos
Integrantes:


In [ ]:
# Imports
import math
import random
from pathlib import Path

In [ ]:

# Rutas relativas a la carpeta desde la que se ejecuta el notebook.
PATH_SOLOMON = Path("data/originales")
PATH_GENERADAS = Path("data/instancias")

ARCHIVO_R108 = PATH_SOLOMON / "R108.txt"
ARCHIVO_R110 = PATH_SOLOMON / "R110.txt"

# Comprobar que existen los originales.
for archivo in [ARCHIVO_R108, ARCHIVO_R110]:
    if not archivo.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo: {archivo.resolve()}"
        )

print("Entrada:", PATH_SOLOMON.resolve())
print("Salida:", PATH_GENERADAS.resolve())

# Sección 1 Lectura de archivos originales

In [ ]:
def leer_solomon(path):
    """Lee cabecera VEHICLE y filas CUSTOMER, sin depender de espacios exactos."""
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    vehicles = capacity = None
    rows = {}
    in_customers = False
    for line in lines:
        s = line.split()
        if not s:
            continue
        if s[0].upper() == "CUSTOMER":
            in_customers = True
            continue
        if not in_customers and len(s) == 2:
            try:
                vehicles, capacity = int(s[0]), float(s[1])
            except ValueError:
                pass
        if in_customers and len(s) == 7:
            try:
                row = tuple(float(x) for x in s)
            except ValueError:
                continue
            i = int(row[0])
            if i != row[0] or i in rows:
                raise ValueError("Identificador no entero o duplicado")
            if row[3] < 0 or row[4] > row[5] or row[6] < 0:
                raise ValueError(f"Datos invalidos en cliente {i}")
            rows[i] = row
    if 0 not in rows or vehicles is None or capacity is None:
        raise ValueError(f"Formato Solomon incompleto: {path}")
    return rows, vehicles, capacity


def distancia(rows, i, j):
    return math.hypot(
        rows[i][1] - rows[j][1], 
        rows[i][2] - rows[j][2]
        )

# Sección 2 Generación de instancias propias

In [ ]:
def generar(source, destination, n, days, vehicles, seed, slack):
    rows, original_vehicles, original_q = leer_solomon(source)
    ids = sorted(i for i in rows if i != 0)[:n] # Solo los primeros n clientes, sin el depot (0).
    # Validaciones de parametros 
    if len(ids) != n or n < 15 or days < 3 or not 3 <= vehicles <= n:
        raise ValueError("Se requieren >=15 clientes, >=3 dias y 3<=vehiculos<=clientes")
    if not 0 <= slack <= 1:
        raise ValueError("El margen de capacidad debe estar entre 0 y 1")

    # Generacion de demanda diaria aleatoria, con al menos un dia activo por cliente.    
    rng = random.Random(seed)
    demand = {}
    for i in ids:
        active = set(rng.sample(range(days), rng.choice([days - 1, days])))
        demand[i] = [max(1, round(rows[i][3] * rng.uniform(0.8, 1.2)))
                     if t in active else 0 for t in range(days)]

    # Agrupacion geografica sencilla para construir UNA solución factible de referencia.
    ordered = sorted(ids, key=lambda i: math.atan2(
        rows[i][2] - rows[0][2], 
        rows[i][1] - rows[0][1]
        ))

    # Equilibrio de numero de clientes     
    groups = [
        ordered[r*n//vehicles:(r+1)*n//vehicles]
        for r in range(vehicles)
    ]

    # Calculo de cargas diarias para cada grupo.
    loads = [
        [sum(demand[i][t] for i in group) for t in range(days)]
             for group in groups
    ]
    # Toma la mayor carga y se redondea hacia arriba con el margen de slack para obtener la nueva capacidad Q.
    q = math.ceil(max(max(x) for x in loads) * (1 + slack))


    # Preparar horarios de apertura y cierre, y ventanas de servicio para cada cliente.
    opening, original_close = rows[0][4:6]
    service = {i: max(1.0, rows[i][6]) for i in ids}
    ready = {i: rows[i][4] for i in ids}
    due = {i: rows[i][5] for i in ids}

    schedules = []
    closing = original_close

    # Construir los recorridos diarios 
    for r, group in enumerate(groups, 1):
        for t in range(days):
            # Recorre cada ruta y cada día. Las rutas se numeran desde 1.
            remaining = {i for i in group if demand[i][t] > 0}
            previous, clock, visits = 0, opening, []
            while remaining:
                # Escoge el siguiente cliente que podria empezar antes. 
                # inicio posible = max{apertura del cliente, hora actual + tiempo de viaje}, si hay empate el identificador mas bajo gana.
                i = min(remaining, key=lambda j: (
                    max(ready[j], clock + distancia(rows, previous, j)), j
                ))
                # Calcula el inicio de servicio y actualiza la ventana de cierre si es necesario.
                start = max(ready[i], clock + distancia(rows, previous, i))
                due[i] = max(due[i], math.ceil(start))

                # Registra la visita, suma el servicio, actualiza la ubicación y elimina al cliente de los pendientes.
                visits.append((i, start))
                clock, previous = start + service[i], i
                remaining.remove(i)

            # Volver al deposito, calcula el regreso y amplía el cierre del depósito si hace falta.
            return_time = clock + distancia(rows, previous, 0)
            closing = max(closing, math.ceil(return_time))
            schedules.append((r, t, visits, return_time))

    # La siguiente parte recorre los horarios construidos y los comprueba.
    # Validacion independiente de la solución factible de referencia: cobertura, consistencia, carga y tiempos.
    for t in range(days):
        seen = []
        for r, day, visits, return_time in schedules:
            if day != t:
                continue
            previous, clock, load = 0, opening, 0
            for i, start in visits:
                if i not in groups[r-1] or demand[i][t] <= 0:
                    raise ValueError("solución factible de referencia inconsistente")
                if not ready[i] <= start <= due[i]:
                    raise ValueError("Ventana incumplida")
                if start + 1e-8 < clock + distancia(rows, previous, i):
                    raise ValueError("Tiempo de viaje incumplido")
                seen.append(i)
                load += demand[i][t]
                previous, clock = i, start + service[i]
            if load > q or return_time > closing:
                raise ValueError("Capacidad u horario incumplido")
            if abs(return_time - clock - distancia(rows, previous, 0)) > 1e-8:
                raise ValueError("Retorno inconsistente")
        expected = sorted(i for i in ids if demand[i][t] > 0)
        if sorted(seen) != expected:
            raise ValueError("Cobertura diaria incorrecta")

    destination.parent.mkdir(parents=True, exist_ok=True)
    name = destination.stem
    text = [f"NAME {name}", "FORMAT CONVRP_MULTIDAY_V1",
            f"SOURCE {source.name}",
            f"SEED {seed}", f"DAYS {days}", f"VEHICLES {vehicles}",
            f"CAPACITY {q}", "DISTANCE EUCLIDEAN_DOUBLE",
            "TRAVEL_TIME EQUAL_DISTANCE", "TIME_UNIT ABSTRACT",
            "", "DEPOT", "ID X Y OPEN CLOSE",
            f"0 {rows[0][1]:g} {rows[0][2]:g} {opening:g} {closing:g}",
            "", "CUSTOMERS", "ID X Y READY DUE SERVICE"]
    for i in ids:
        text.append(f"{i} {rows[i][1]:g} {rows[i][2]:g} "
                    f"{ready[i]:g} {due[i]:g} {service[i]:g}")
    text += ["", "DAILY_DEMAND", "ID " + " ".join(f"DAY{t+1}" for t in range(days))]
    text += [f"{i} " + " ".join(map(str, demand[i])) for i in ids]
    text += ["", "EOF"]
    destination.write_text("\n".join(text) + "\n", encoding="utf-8")

    widened = [i for i in ids if due[i] != rows[i][5]]
    report = [f"Instancia: {name}", "solución factible de referencia VERIFICADO (no es una solucion optima)",
              "Dificultad computacional: PENDIENTE de medir con el solver.",
              f"Vehiculos originales: {original_vehicles}; nuevos: {vehicles}",
              f"Capacidad original: {original_q:g}; nueva: {q}; margen: {slack:.1%}",
              f"Cierre original: {original_close:g}; nuevo: {closing:g}",
              f"Ventanas con limite superior ampliado: {len(widened)}", ""]
    for i in widened:
        report.append(f"Cliente {i}: DUE {rows[i][5]:g} -> {due[i]:g}")
    for r, group in enumerate(groups, 1):
        report += [f"\nAsignacion solución factible de referencia ruta {r}: {group}", f"Cargas: {loads[r-1]}"]
    for r, t, visits, return_time in schedules:
        route = " -> ".join(["0"] + [str(i) for i, _ in visits] + ["0"])
        starts = ", ".join(f"{i}: {w:.6f}" for i, w in visits)
        report += [f"\nRuta {r}, dia {t+1}: {route}",
                   f"Salida fija: {opening:g}; retorno: {return_time:.6f}",
                   f"Inicio de servicio (valores mostrados redondeados): {starts}"]
    destination.with_name(name + "_verificacion.txt").write_text(
        "\n".join(report) + "\n", encoding="utf-8")
    print(f"{destination.name}: {n} clientes, {days} dias, {vehicles} vehiculos, "
          f"Q={q}, cierre={closing:g}, ventanas ampliadas={len(widened)}; solución factible de referencia OK")

In [ ]:
ARCHIVO_I1 = PATH_GENERADAS / "I1.txt"
ARCHIVO_I2 = PATH_GENERADAS / "I2.txt"

generar(
    source=ARCHIVO_R108,
    destination=ARCHIVO_I1,
    n=20,
    days=3,
    vehicles=3,
    seed=42,
    slack=0.05,
)

generar(
    source=ARCHIVO_R110,
    destination=ARCHIVO_I2,
    n=30,
    days=3,
    vehicles=3,
    seed=43,
    slack=0.05,
)

# Sección 3 Lector de instancias propias

In [ ]:
def leer_multidia(path):
    """Lee el formato CONVRP_MULTIDAY_V1 generado por nuestro código."""
    path = Path(path)

    cabecera = {}
    clientes = {}
    demandas = {}
    deposito = None
    seccion = None

    for numero_linea, linea in enumerate(
        path.read_text(encoding="utf-8-sig").splitlines(), start=1
    ):
        linea = linea.strip()

        if not linea or linea.startswith("#"):
            continue

        if linea == "EOF":
            break

        if linea in {"DEPOT", "CUSTOMERS", "DAILY_DEMAND"}:
            seccion = linea
            continue

        partes = linea.split()

        # Ignorar títulos de columnas.
        if partes[0] == "ID":
            continue

        try:
            if seccion is None:
                clave, valor = linea.split(maxsplit=1)
                cabecera[clave] = valor

            elif seccion == "DEPOT":
                if len(partes) != 5 or deposito is not None:
                    raise ValueError("Debe existir una única fila de depósito")

                identificador = int(partes[0])
                x, y, apertura, cierre = map(float, partes[1:])

                deposito = {
                    "id": identificador,
                    "x": x,
                    "y": y,
                    "apertura": apertura,
                    "cierre": cierre,
                }

            elif seccion == "CUSTOMERS":
                if len(partes) != 6:
                    raise ValueError("Fila de cliente con columnas incorrectas")

                i = int(partes[0])

                if i in clientes:
                    raise ValueError(f"Cliente duplicado: {i}")

                x, y, inicio, fin, servicio = map(float, partes[1:])

                clientes[i] = {
                    "x": x,
                    "y": y,
                    "inicio": inicio,
                    "fin": fin,
                    "servicio": servicio,
                }

            elif seccion == "DAILY_DEMAND":
                i = int(partes[0])

                if i in demandas:
                    raise ValueError(f"Demanda duplicada para cliente {i}")

                demandas[i] = list(map(float, partes[1:]))

        except ValueError as error:
            raise ValueError(
                f"{path.name}, línea {numero_linea}: {error}"
            ) from error

    # Comprobaciones de estructura.
    if cabecera.get("FORMAT") != "CONVRP_MULTIDAY_V1":
        raise ValueError("Formato multidía no reconocido")

    dias = int(cabecera["DAYS"])
    vehiculos = int(cabecera["VEHICLES"])
    capacidad = float(cabecera["CAPACITY"])

    if dias <= 0 or vehiculos <= 0 or capacidad <= 0:
        raise ValueError("Días, vehículos y capacidad deben ser positivos")

    if deposito is None or deposito["id"] != 0:
        raise ValueError("Falta el depósito con identificador 0")

    if not 0 <= deposito["apertura"] <= deposito["cierre"]:
        raise ValueError("Horario del depósito inválido")

    n = len(clientes)

    # Nuestros archivos generados seleccionan los clientes 1,...,n.
    if n == 0 or set(clientes) != set(range(1, n + 1)):
        raise ValueError("Se esperan clientes numerados desde 1 hasta n")

    if set(demandas) != set(clientes):
        raise ValueError("Los clientes y las filas de demanda no coinciden")

    for i in clientes:
        c = clientes[i]

        if not 0 <= c["inicio"] <= c["fin"]:
            raise ValueError(f"Ventana inválida del cliente {i}")

        if c["servicio"] <= 0:
            raise ValueError(f"Servicio no positivo del cliente {i}")

        if len(demandas[i]) != dias:
            raise ValueError(f"Cantidad de días incorrecta del cliente {i}")

        if any(q < 0 for q in demandas[i]):
            raise ValueError(f"Demanda negativa del cliente {i}")

    if (
        cabecera.get("DISTANCE") != "EUCLIDEAN_DOUBLE"
        or cabecera.get("TRAVEL_TIME") != "EQUAL_DISTANCE"
    ):
        raise ValueError("Convención de distancia o tiempo no compatible")

    # Conjuntos del modelo.
    C = list(range(1, n + 1))
    D = list(range(1, dias + 1))
    R = list(range(1, vehiculos + 1))

    retorno = n + 1
    N = [0] + C + [retorno]

    # Todos los clientes tienen ventana en nuestro formato actual.
    N_TW = C.copy()

    # Depósito de salida y retorno: misma ubicación física.
    coordenadas = {
        0: (deposito["x"], deposito["y"]),
        retorno: (deposito["x"], deposito["y"]),
    }
    coordenadas.update({
        i: (clientes[i]["x"], clientes[i]["y"]) for i in C
    })

    td = {
        (i, j): math.hypot(
            coordenadas[i][0] - coordenadas[j][0],
            coordenadas[i][1] - coordenadas[j][1],
        )
        for i in N
        for j in N
    }

    tt = td.copy()

    q = {
        (i, t): demandas[i][t - 1]
        for i in C
        for t in D
    }

    O = [(i, t) for i in C for t in D if q[i, t] > 0]

    st = {
        (i, t): clientes[i]["servicio"]
        for i, t in O
    }

    # Arcos diarios: solo clientes con pedido ese día.
    A = []

    for t in D:
        activos = [i for i in C if q[i, t] > 0]

        A.append((0, retorno, t))

        A.extend((0, i, t) for i in activos)
        A.extend((i, retorno, t) for i in activos)

        A.extend(
            (i, j, t)
            for i in activos
            for j in activos
            if i != j
        )

    return {
        "nombre": cabecera.get("NAME", path.stem),
        "C": C,
        "D": D,
        "R": R,
        "N": N,
        "N_TW": N_TW,
        "O": O,
        "A": A,
        "retorno": retorno,
        "coordenadas": coordenadas,
        "td": td,
        "tt": tt,
        "q": q,
        "st": st,
        "a": {i: clientes[i]["inicio"] for i in C},
        "b": {i: clientes[i]["fin"] for i in C},
        "Q": capacidad,
        "aW": deposito["apertura"],
        "bW": deposito["cierre"],
    }

In [ ]:
# Leer desde la carpeta de instancias generadas.
instancia_1 = leer_multidia(PATH_GENERADAS / "I1.txt")
instancia_2 = leer_multidia(PATH_GENERADAS / "I2.txt")

instancias = {
    "I1": instancia_1,
    "I2": instancia_2,
}

for identificador, datos in instancias.items():
    print(
        f"{identificador}: "
        f"{len(datos['C'])} clientes, "
        f"{len(datos['D'])} días, "
        f"{len(datos['R'])} vehículos, "
        f"capacidad {datos['Q']}"
    )